In [2]:
# %% Shared setup (standard library only - NO pandas / NumPy)
import csv, json, shutil
from pathlib import Path
from collections import Counter, defaultdict

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"          # put the Kaggle CSV files here (git-ignored)
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for d in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    d.mkdir(exist_ok=True)
print("Seed:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Name Length:", NAME_LENGTH)

# %% EXERCISE 2 - Mobile Device Usage Profiling (user_behavior_dataset.csv)
mob_path = DATA_DIR / "user_behavior_dataset.csv"
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_CLASS = (SEED_DIGIT % 5) + 1
FIELDS = ["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"]

total_rows = os_rows = 0
chosen = []
with open(mob_path, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_rows += 1
        if row["Operating System"].strip() != TARGET_OS:
            continue
        os_rows += 1
        try:
            if int(row["User Behavior Class"]) == TARGET_CLASS:
                chosen.append({k: row[k] for k in FIELDS})
        except (ValueError, KeyError):
            continue

if chosen:
    avg_batt = sum(float(r["Battery Drain (mAh/day)"]) for r in chosen) / len(chosen)
    avg_data = sum(float(r["Data Usage (MB/day)"]) for r in chosen) / len(chosen)
else:
    avg_batt = avg_data = 0.0                      # no matching records

export_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_users.csv"
with open(export_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(chosen)
with open(export_csv, "r", encoding="utf-8", newline="") as f:
    exported_rows = sum(1 for _ in csv.DictReader(f))

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({"operating_system": TARGET_OS, "behavior_class": TARGET_CLASS, "selected_records": len(chosen),
               "exported_rows": exported_rows, "avg_battery_drain_mAh_day": round(avg_batt, 2),
               "avg_data_usage_MB_day": round(avg_data, 2)}, f, indent=4)
backup_csv = BACKUP_DIR / export_csv.name
shutil.copy2(export_csv, backup_csv)

print("Operating system and behavior class:", TARGET_OS, "/ class", TARGET_CLASS)
print("Total records in dataset:", total_rows)
print("Records matching operating system:", os_rows)
print("Records in target behavior class:", len(chosen))
print("Average battery drain (mAh/day):", round(avg_batt, 2))
print("Average data usage (MB/day):", round(avg_data, 2))
print("Count verification:", exported_rows == len(chosen))
print("Exported CSV file:", export_csv)
print("JSON summary file:", json_path)
print("Backup file and size:", backup_csv.name, backup_csv.stat().st_size, "bytes")


Seed: 2 | ID Sum: 23 | Name Length: 9
Operating system and behavior class: Android / class 3
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 111
Average battery drain (mAh/day): 1521.69
Average data usage (MB/day): 820.26
Count verification: True
Exported CSV file: output\Tapiculin_TUPM-26-1932_mobile_users.csv
JSON summary file: output\Tapiculin_TUPM-26-1932_mobile_summary.json
Backup file and size: Tapiculin_TUPM-26-1932_mobile_users.csv 4068 bytes
